#MAKING A SILVER TABLE FOR REVIEWS

In [0]:
df = spark.table("steam_data.default.review")
df.printSchema()
print(len(df.columns), "columns")
print(df.dtypes)
n_rows = df.count()
print("NUMBER OF ROWS:", n_rows)

In [0]:
    %sql
    Select * from steam_data.default.review 

REMOVING NULL VALUES AND REVIEWS WITH LENGTH LESS THAN 5

In [0]:
%sql
CREATE OR REPLACE TABLE steam_data.default.silver_reviews AS
SELECT *
FROM steam_data.default.review
WHERE reviews IS NOT NULL
  AND LENGTH(TRIM(reviews)) >= 5

##CHECK FOR DUPLICATE ROWS

In [0]:
%sql
SELECT COUNT(*) AS total,
       COUNT(DISTINCT app_id, reviews) AS distinct_app_review
FROM steam_data.default.silver_reviews;

Only 18 of them found,we can still remove them to imporve quality of our analysis

In [0]:
%sql
CREATE OR REPLACE TABLE steam_data.default.silver_reviews AS
SELECT DISTINCT *
FROM steam_data.default.silver_reviews;

In [0]:
silv_df=spark.table("steam_data.default.silver_reviews")

n_rows_silv = silv_df.count()
print("NUMBER OF ROWS IN THE GIVEN TABLE:" , n_rows)

print("NUMBER OF ROWS AFTER REMOVING DUPLICATES,NULLS:", n_rows_silv)

print("NUMBER OF ROWS DELETED:", n_rows - n_rows_silv)



##CLEANING THE REVIEW COLUMN    

In [0]:
%sql
CREATE OR REPLACE TABLE steam_data.default.silver_reviews AS
SELECT
    app_id,
    name,
    reviews,
    TRIM(REGEXP_REPLACE(REGEXP_REPLACE(reviews, '\\[[^\\]]*\\]', ' '), '\\s+', ' ')) AS review_clean
FROM steam_data.default.silver_reviews;

In [0]:
%sql
DELETE FROM steam_data.default.silver_reviews
WHERE review_clean IS NULL OR review_length < 10;

Deduping the table after removing special characters and spaces

In [0]:
%sql
CREATE OR REPLACE TABLE steam_data.default.silver_reviews AS
SELECT app_id,FIRST(name) AS name,FIRST(reviews) AS reviews,review_clean, length(review_clean) AS review_length
FROM steam_data.default.silver_reviews
GROUP BY app_id,review_clean
    


In [0]:
%sql
SELECT COUNT(*) AS total_reviews,
       COUNT(DISTINCT app_id) AS games_with_reviews,
       ROUND(AVG(review_length)) AS avg_length,
       PERCENTILE(review_length, 0.5) AS median_length,
       MAX(review_length) AS max_length
FROM steam_data.default.silver_reviews;

#HANDLING THE LANGUAGE OF THE REVIEW    

In [0]:
%pip install langdetect

In [0]:
%restart_python

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # makes results repeatable

@pandas_udf(StringType())
def detect_lang(texts: pd.Series) -> pd.Series:
    def safe(t):
        try:
            return detect(t)
        except Exception:
            return "unknown"
    return texts.apply(safe)

df = spark.table("steam_data.default.silver_reviews")

with_lang = df.withColumn("lang", detect_lang(F.col("review_clean")))

with_lang.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("steam_data.default.silver_reviews_lang")

In [0]:
%sql
select lang,count(*) from steam_data.default.silver_reviews_lang group by lang order by count(*) desc

From above table it is obvious english is more than 95% of the table , so its better to drop all the other languages
    

In [0]:
%sql
CREATE OR REPLACE TABLE steam_data.default.silver_reviews_en AS
SELECT *
FROM steam_data.default.silver_reviews_lang
WHERE lang = 'en';      

In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM steam_data.default.silver_reviews_lang) AS before_filter,
    (SELECT COUNT(*) FROM steam_data.default.silver_reviews_en)   AS english_only;

Deleted 446 rows which were not in english 

In [0]:
%sql
select * from steam_data.default.silver_reviews_en

Some rows have a ambigous app id, I am removing them to avoid errors

In [0]:
%sql
SELECT COUNT(*) AS bad_rows
FROM steam_data.default.silver_reviews_en
WHERE app_id IS NOT NULL AND try_cast(app_id AS BIGINT) IS NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE steam_data.default.silver_reviews_en AS
SELECT try_cast(app_id AS BIGINT) AS app_id,
       name,
       reviews,
       review_clean,
       review_length,
       lang
FROM steam_data.default.silver_reviews_en
WHERE try_cast(app_id AS BIGINT) IS NOT NULL;

Checking for games which are not in games table but in review table

In [0]:
%sql
SELECT COUNT(DISTINCT r.app_id) AS review_games_not_in_games,
       COUNT(*) AS orphan_reviews
FROM steam_data.default.silver_reviews_en r
LEFT ANTI JOIN steam_data.default.silver_games g
  ON r.app_id = g.app_id;

In [0]:
%sql
SELECT COUNT(*) AS total_reviews,
       COUNT(DISTINCT app_id) AS games_with_reviews,
       ROUND(AVG(review_length)) AS avg_length,
       PERCENTILE(review_length, 0.5) AS median_length,
       MAX(review_length) AS max_length
FROM steam_data.default.silver_reviews_en;